In [ ]:
DATA_PATH = "data"
import sys
sys.path.append(DATA_PATH)

In [ ]:
import polars as pl
import pandas as pd
import gc
from datetime import datetime


# --- Lazy Okuma Fonksiyonu ---
def lazy_load_parquet(path):
    """Parquet dosyasını lazy bir şekilde okur."""
    return pl.scan_parquet(f"{DATA_PATH}/{path}")

# --- Zaman Bazlı Özellikler için Yardımcı Fonksiyonlar ---
def add_hourly_features(df, timestamp_col):
    """
    Saat bazlı verilere mevsimsel özellikler olmadan zaman özellikleri ekler.
    """
    return df.with_columns([
        pl.col(timestamp_col).dt.hour().alias("hour_of_day"),
        pl.col(timestamp_col).dt.weekday().alias("day_of_week"),
        pl.col(timestamp_col).dt.day().alias("day_of_month"),
        pl.col(timestamp_col).dt.month().alias("month_of_year"),
        pl.col(timestamp_col).dt.date().alias("date_only"),
        
        # Günün bölümleri
        pl.when((pl.col(timestamp_col).dt.hour() >= 5) & (pl.col(timestamp_col).dt.hour() < 12)).then(1)  # Morning (5-12)
        .when((pl.col(timestamp_col).dt.hour() >= 12) & (pl.col(timestamp_col).dt.hour() < 17)).then(2)   # Afternoon (12-17)
        .when((pl.col(timestamp_col).dt.hour() >= 17) & (pl.col(timestamp_col).dt.hour() < 21)).then(3)   # Evening (17-21)
        .otherwise(4)                                                                                    # Night (21-5)
        .alias("time_of_day"),
        
        # Hafta sonu/hafta içi
        pl.when((pl.col(timestamp_col).dt.weekday() == 5) | (pl.col(timestamp_col).dt.weekday() == 6))
        .then(1)
        .otherwise(0)
        .alias("is_weekend")
    ])

def add_daily_features(df, date_col):
    """
    Gün bazlı verilere mevsimsel özellikler olmadan zaman özellikleri ekler.
    """
    return df.with_columns([
        pl.col(date_col).dt.weekday().alias("day_of_week"),
        pl.col(date_col).dt.day().alias("day_of_month"),
        pl.col(date_col).dt.month().alias("month_of_year"),
        
        # Hafta sonu/hafta içi
        pl.when((pl.col(date_col).dt.weekday() == 5) | (pl.col(date_col).dt.weekday() == 6))
        .then(1)
        .otherwise(0)
        .alias("is_weekend")
    ])
    
# --- Zaman Bazlı Oranlar ---
def calculate_time_ratios(df, date_col, group_cols, value_cols, is_hourly=False):
    """
    Zaman bazlı oranları hesaplar
    - Son 1/3/7/14 günlük aktivitelerin toplam aktiviteye oranı
    - Hafta sonu/hafta içi oranları
    - Günün bölümlerine göre oranlar (sadece saatlik verilerde)
    """
    df_processed = df.with_columns(
        pl.col(date_col).cast(pl.Datetime(time_unit="ms")).alias('processed_date')
    )
    
    # Toplam aktivite
    total_agg = df_processed.group_by(group_cols).agg([
        pl.sum(col).alias(f"{col}_total") for col in value_cols
    ])
    
    # Son periyotlar
    periods = {
        '1d': pl.duration(days=1),
        '3d': pl.duration(days=3),
        '7d': pl.duration(days=7),
        '14d': pl.duration(days=14),
        'weekend': None,
        'weekday': None
    }
    
    if is_hourly:
        periods.update({
            'morning': None,
            'afternoon': None,
            'evening': None,
            'night': None
        })
    
    period_aggs = []
    for period_name, period_duration in periods.items():
        if period_name in ['weekend', 'weekday']:
            if period_name == 'weekend':
                filter_expr = (pl.col('processed_date').dt.weekday() >= 5)
            else:
                filter_expr = (pl.col('processed_date').dt.weekday() < 5)
            period_agg = df_processed.filter(filter_expr).group_by(group_cols).agg([
                pl.sum(col).alias(f"{col}_{period_name}") for col in value_cols
            ])
        elif period_name in ['morning', 'afternoon', 'evening', 'night']:
            hour_ranges = {
                'morning': (5, 12),
                'afternoon': (12, 17),
                'evening': (17, 21),
                'night': (21, 5)
            }
            start, end = hour_ranges[period_name]
            if period_name == 'night':
                filter_expr = (pl.col('processed_date').dt.hour() >= start) | (pl.col('processed_date').dt.hour() < end)
            else:
                filter_expr = (pl.col('processed_date').dt.hour() >= start) & (pl.col('processed_date').dt.hour() < end)
            
            period_agg = df_processed.filter(filter_expr).group_by(group_cols).agg([
                pl.sum(col).alias(f"{col}_{period_name}") for col in value_cols
            ])
        else:
            start_date = df_processed.select(pl.max('processed_date')).collect().item() - period_duration
            period_agg = df_processed.filter(
                pl.col('processed_date') >= start_date
            ).group_by(group_cols).agg([
                pl.sum(col).alias(f"{col}_{period_name}") for col in value_cols
            ])
        
        period_aggs.append(period_agg)
    
    # Tüm aggregasyonları birleştir
    merged_df = total_agg
    for period_agg in period_aggs:
        merged_df = merged_df.join(period_agg, on=group_cols, how='left')
    
    # Oranları hesapla
    for col in value_cols:
        # Son periyotların toplama oranı
        for period in ['1d', '3d', '7d', '14d']:
            merged_df = merged_df.with_columns(
                (pl.col(f"{col}_{period}") / pl.col(f"{col}_total")).fill_nan(0).alias(f"{col}_{period}_to_total_ratio")
            )
        
        # Hafta sonu/hafta içi oranları
        merged_df = merged_df.with_columns(
            (pl.col(f"{col}_weekend") / pl.col(f"{col}_weekday")).fill_nan(0).alias(f"{col}_weekend_to_weekday_ratio")
        )
        
        if is_hourly:
            # Günün bölümleri oranları
            merged_df = merged_df.with_columns(
                (pl.col(f"{col}_morning") / pl.col(f"{col}_total")).fill_nan(0).alias(f"{col}_morning_ratio"),
                (pl.col(f"{col}_afternoon") / pl.col(f"{col}_total")).fill_nan(0).alias(f"{col}_afternoon_ratio"),
                (pl.col(f"{col}_evening") / pl.col(f"{col}_total")).fill_nan(0).alias(f"{col}_evening_ratio"),
                (pl.col(f"{col}_night") / pl.col(f"{col}_total")).fill_nan(0).alias(f"{col}_night_ratio")
            )
    
    return merged_df

# --- Aktivite Yoğunluğu Özellikleri ---
def calculate_activity_density(df, date_col, group_cols, is_hourly=False):
    """
    Aktivite yoğunluğu özellikleri hesaplar
    - Aktif olduğu gün/saat sayısı
    - Toplam gün/saat sayısına oranı
    """
    if is_hourly:
        return df.with_columns(
            pl.col(date_col).dt.date().alias('activity_date'),
            pl.col(date_col).dt.hour().alias('activity_hour')
        ).group_by(group_cols).agg([
            pl.n_unique('activity_date').alias('active_days_count'),
            pl.n_unique('activity_hour').alias('active_hours_count'),
            (pl.n_unique('activity_date') / 35).alias('daily_activity_density'),  # 35 günlük veri
            (pl.n_unique('activity_hour') / (35*24)).alias('hourly_activity_density')
        ])
    else:
        return df.with_columns(
            pl.col(date_col).dt.date().alias('activity_date')
        ).group_by(group_cols).agg([
            pl.n_unique('activity_date').alias('active_days_count'),
            (pl.n_unique('activity_date') / 35).alias('daily_activity_density')  # 35 günlük veri
        ])

# --- Son Etkileşim Özellikleri ---
def calculate_last_interaction_features(df, date_col, group_cols):
    """
    Son etkileşim özellikleri hesaplar
    - Son etkileşim tarihi
    - Son etkileşimden bu yana geçen gün sayısı
    """
    max_date = df.select(pl.max(date_col)).collect().item()
    return df.group_by(group_cols).agg([
        pl.max(date_col).alias('last_interaction_date'),
        (max_date - pl.max(date_col)).dt.days().alias('days_since_last_interaction')
    ])

# --- Statik Zaman Bazlı Oranları Hesaplama Fonksiyonu ---
def calculate_overall_ratios_pl(df, date_col, group_by_cols, value_cols):
    """
    Polars'ta tüm veri setine bakarak statik 3/7 ve 3/30 günlük oranları hesaplar.
    Ayrıca 7/30 günlük oranları da hesaplar.
    """
    df_processed = df.with_columns(
        pl.col(date_col).cast(pl.Datetime(time_unit="ms")).alias('processed_date')
    )
    max_date = df_processed.select(pl.col('processed_date').max()).collect().item()
    recent_3d_start = max_date - pl.duration(days=3)
    recent_7d_start = max_date - pl.duration(days=7)
    recent_30d_start = max_date - pl.duration(days=30)

    recent_3d_agg = df_processed.filter(
        pl.col('processed_date') >= recent_3d_start
    ).group_by(group_by_cols).agg([
        pl.sum(col).alias(f"{col}_3d_sum") for col in value_cols
    ])
    
    recent_7d_agg = df_processed.filter(
        pl.col('processed_date') >= recent_7d_start
    ).group_by(group_by_cols).agg([
        pl.sum(col).alias(f"{col}_7d_sum") for col in value_cols
    ])
    
    recent_30d_agg = df_processed.filter(
        pl.col('processed_date') >= recent_30d_start
    ).group_by(group_by_cols).agg([
        pl.sum(col).alias(f"{col}_30d_sum") for col in value_cols
    ])
    
    # Tüm aggregasyonları birleştir
    merged_df = recent_7d_agg.join(recent_30d_agg, on=group_by_cols, how='left')
    merged_df = merged_df.join(recent_3d_agg, on=group_by_cols, how='left')
    
    # Oranları hesapla ve ekle
    for col in value_cols:
        merged_df = merged_df.with_columns(
            (pl.col(f"{col}_3d_sum") / pl.col(f"{col}_7d_sum")).fill_nan(0).alias(f'{col}_3d_to_7d_ratio'),
            (pl.col(f"{col}_3d_sum") / pl.col(f"{col}_30d_sum")).fill_nan(0).alias(f'{col}_3d_to_30d_ratio'),
            (pl.col(f"{col}_7d_sum") / pl.col(f"{col}_30d_sum")).fill_nan(0).alias(f'{col}_7d_to_30d_ratio')
        )
    
    # Sadece oran sütunlarını ve group by sütunlarını döndür
    return merged_df.select(group_by_cols + 
                            [f'{col}_3d_to_7d_ratio' for col in value_cols] +
                            [f'{col}_3d_to_30d_ratio' for col in value_cols] +
                            [f'{col}_7d_to_30d_ratio' for col in value_cols]
                           )

# --- Fiyat Özelliklerinden Oranları Hesaplama Fonksiyonu ---
def add_price_ratios_pl(df):
    """
    Polars DataFrame'ine fiyat verilerinden anlamlı oranlar ekler.
    """
    return df.with_columns([
        ((pl.col("content_daily_original_price_mean") - pl.col("content_daily_selling_price_mean")) / pl.col("content_daily_original_price_mean")).fill_nan(0).alias("discount_rate"),
        (pl.col("content_daily_selling_price_mean") / pl.col("content_daily_original_price_mean")).fill_nan(0).alias("selling_to_original_price_ratio"),
        (pl.col("content_daily_discounted_price_mean") / pl.col("content_daily_selling_price_mean")).fill_nan(0).alias("discounted_to_selling_price_ratio"),
        (pl.col("content_daily_content_rate_avg_mean") / pl.col("content_daily_selling_price_mean")).fill_nan(0).alias("rate_to_price_ratio"),
        (pl.col("content_daily_rate_count_mean") / pl.col("content_daily_review_count_mean")).fill_nan(0).alias("rate_to_review_count_ratio")
    ])

# --- Veri Yükleme ---
print("Veri setleri yükleniyor...")
content_metadata = lazy_load_parquet("content/metadata.parquet").collect()
content_price_data_lazy = lazy_load_parquet("content/price_rate_review_data.parquet")
content_search_log_lazy = lazy_load_parquet("content/search_log.parquet")
content_sitewide_log_lazy = lazy_load_parquet("content/sitewide_log.parquet")
content_top_terms_log_lazy = lazy_load_parquet("content/top_terms_log.parquet")
user_metadata = lazy_load_parquet("user/metadata.parquet").collect()
user_search_log_lazy = lazy_load_parquet("user/search_log.parquet")
user_sitewide_log_lazy = lazy_load_parquet("user/sitewide_log.parquet")
user_top_terms_log_lazy = lazy_load_parquet("user/top_terms_log.parquet")
user_fashion_search_log_lazy = lazy_load_parquet("user/fashion_search_log.parquet")
user_fashion_sitewide_log_lazy = lazy_load_parquet("user/fashion_sitewide_log.parquet")
term_search_log_lazy = lazy_load_parquet("term/search_log.parquet")

# --- Zamansal Özelliklerin Eklenmesi ---
print("Zamansal özellikler ekleniyor...")
hourly_datasets = {
    'user_search': user_search_log_lazy,
    'user_sitewide': user_sitewide_log_lazy,
    'user_top_terms': user_top_terms_log_lazy,
    'user_fashion_search': user_fashion_search_log_lazy,
    'user_fashion_sitewide': user_fashion_sitewide_log_lazy,
    'term_search': term_search_log_lazy
}
for name, dataset in hourly_datasets.items():
    hourly_datasets[name] = add_hourly_features(dataset, 'ts_hour')

user_search_log_lazy, user_sitewide_log_lazy, user_top_terms_log_lazy, \
user_fashion_search_log_lazy, user_fashion_sitewide_log_lazy, term_search_log_lazy = hourly_datasets.values()

daily_datasets = {
    'content_search': content_search_log_lazy,
    'content_sitewide': content_sitewide_log_lazy,
    'content_top_terms': content_top_terms_log_lazy
}
for name, dataset in daily_datasets.items():
    daily_datasets[name] = add_daily_features(dataset, 'date')

content_search_log_lazy, content_sitewide_log_lazy, content_top_terms_log_lazy = daily_datasets.values()

# --- Statik Özellik Mühendisliği (Yeni oranlar eklendi) ---
print("Statik özellikler oluşturuluyor...")
term_popularity_agg = term_search_log_lazy.group_by('search_term_normalized').agg(
    pl.sum('total_search_impression').alias('term_total_impression_sum'),
    pl.mean('total_search_impression').alias('term_total_impression_mean'),
    pl.sum('total_search_click').alias('term_total_click_sum'),
    pl.mean('total_search_click').alias('term_total_click_mean')
).with_columns(
    (pl.col('term_total_click_sum') / pl.col('term_total_impression_sum')).fill_nan(0).alias('term_ctr_sum'),
    (pl.col('term_total_click_mean') / pl.col('term_total_impression_mean')).fill_nan(0).alias('term_ctr_mean')
).collect()

content_term_stats = content_top_terms_log_lazy.group_by('content_id_hashed').agg(
    pl.sum('total_search_impression').alias('content_term_impression_sum'),
    pl.mean('total_search_impression').alias('content_term_impression_mean'),
    pl.max('total_search_impression').alias('content_term_impression_max'),
    pl.sum('total_search_click').alias('content_term_click_sum'),
    pl.mean('total_search_click').alias('content_term_click_mean'),
    pl.max('total_search_click').alias('content_term_click_max'),
    pl.n_unique('search_term_normalized').alias('content_unique_terms_count')
).with_columns(
    (pl.col('content_term_click_sum') / pl.col('content_term_impression_sum')).fill_nan(0).alias('content_term_ctr_sum'),
    (pl.col('content_term_click_mean') / pl.col('content_term_impression_mean')).fill_nan(0).alias('content_term_ctr_mean')
).collect()

content_search_agg = content_search_log_lazy.group_by('content_id_hashed').agg(
    pl.sum('total_search_impression').alias('content_total_search_impression_sum'),
    pl.mean('total_search_impression').alias('content_total_search_impression_mean'),
    pl.sum('total_search_click').alias('content_total_search_click_sum'),
    pl.mean('total_search_click').alias('content_total_search_click_mean')
).with_columns(
    (pl.col('content_total_search_click_sum') / pl.col('content_total_search_impression_sum')).fill_nan(0).alias('content_search_ctr_sum'),
    (pl.col('content_total_search_click_mean') / pl.col('content_total_search_impression_mean')).fill_nan(0).alias('content_search_ctr_mean')
).collect()

content_sitewide_agg = content_sitewide_log_lazy.group_by('content_id_hashed').agg(
    pl.sum('total_click').alias('content_total_click_sum'),
    pl.mean('total_click').alias('content_total_click_mean'),
    pl.sum('total_cart').alias('content_total_cart_sum'),
    pl.mean('total_cart').alias('content_total_cart_mean'),
    pl.sum('total_fav').alias('content_total_fav_sum'),
    pl.mean('total_fav').alias('content_total_fav_mean'),
    pl.sum('total_order').alias('content_total_order_sum'),
    pl.mean('total_order').alias('content_total_order_mean')
).with_columns(
    (pl.col('content_total_cart_sum') / pl.col('content_total_click_sum')).fill_nan(0).alias('content_cart_to_click_ratio_sum'),
    (pl.col('content_total_cart_mean') / pl.col('content_total_click_mean')).fill_nan(0).alias('content_cart_to_click_ratio_mean'),
    (pl.col('content_total_order_sum') / pl.col('content_total_cart_sum')).fill_nan(0).alias('content_order_to_cart_ratio_sum'),
    (pl.col('content_total_order_mean') / pl.col('content_total_cart_mean')).fill_nan(0).alias('content_order_to_cart_ratio_mean'),
    (pl.col('content_total_fav_sum') / pl.col('content_total_click_sum')).fill_nan(0).alias('content_fav_to_click_ratio_sum'),
    (pl.col('content_total_fav_mean') / pl.col('content_total_click_mean')).fill_nan(0).alias('content_fav_to_click_ratio_mean')
).collect()

user_metadata = user_metadata.with_columns(
    (2025 - pl.col('user_birth_year')).alias('user_age')
)

user_search_agg = user_search_log_lazy.group_by('user_id_hashed').agg(
    pl.sum('total_search_impression').alias('user_total_search_impression_sum'),
    pl.mean('total_search_impression').alias('user_total_search_impression_mean'),
    pl.sum('total_search_click').alias('user_total_search_click_sum'),
    pl.mean('total_search_click').alias('user_total_search_click_mean')
).with_columns(
    (pl.col('user_total_search_click_sum') / pl.col('user_total_search_impression_sum')).fill_nan(0).alias('user_search_ctr_sum'),
    (pl.col('user_total_search_click_mean') / pl.col('user_total_search_impression_mean')).fill_nan(0).alias('user_search_ctr_mean')
).collect()

user_sitewide_agg = user_sitewide_log_lazy.group_by('user_id_hashed').agg(
    pl.sum('total_click').alias('user_total_click_sum'),
    pl.mean('total_click').alias('user_total_click_mean'),
    pl.sum('total_cart').alias('user_total_cart_sum'),
    pl.mean('total_cart').alias('user_total_cart_mean'),
    pl.sum('total_fav').alias('user_total_fav_sum'),
    pl.mean('total_fav').alias('user_total_fav_mean'),
    pl.sum('total_order').alias('user_total_order_sum'),
    pl.mean('total_order').alias('user_total_order_mean')
).with_columns(
    (pl.col('user_total_cart_sum') / pl.col('user_total_click_sum')).fill_nan(0).alias('user_cart_to_click_ratio_sum'),
    (pl.col('user_total_cart_mean') / pl.col('user_total_click_mean')).fill_nan(0).alias('user_cart_to_click_ratio_mean'),
    (pl.col('user_total_order_sum') / pl.col('user_total_cart_sum')).fill_nan(0).alias('user_order_to_cart_ratio_sum'),
    (pl.col('user_total_order_mean') / pl.col('user_total_cart_mean')).fill_nan(0).alias('user_order_to_cart_ratio_mean'),
    (pl.col('user_total_fav_sum') / pl.col('user_total_click_sum')).fill_nan(0).alias('user_fav_to_click_ratio_sum'),
    (pl.col('user_total_fav_mean') / pl.col('user_total_click_mean')).fill_nan(0).alias('user_fav_to_click_ratio_mean')
).collect()

user_top_terms_agg = (
    user_top_terms_log_lazy
    .group_by(["user_id_hashed", "search_term_normalized"])  
    .agg([
        pl.sum("total_search_impression").alias("user_term_search_impression_sum"),
        pl.mean("total_search_impression").alias("user_term_search_impression_mean"),
        pl.sum("total_search_click").alias("user_term_search_click_sum"),
        pl.mean("total_search_click").alias("user_term_search_click_mean"),
    ])
    .with_columns(
        (pl.col('user_term_search_click_sum') / pl.col('user_term_search_impression_sum')).fill_nan(0).alias('user_term_ctr_sum'),
        (pl.col('user_term_search_click_mean') / pl.col('user_term_search_impression_mean')).fill_nan(0).alias('user_term_ctr_mean')
    )
    .collect()
)

user_fashion_search_agg = user_fashion_search_log_lazy.group_by('user_id_hashed').agg(
    pl.sum('total_search_impression').alias('user_fashion_search_impression_sum'),
    pl.mean('total_search_impression').alias('user_fashion_search_impression_mean'),
    pl.sum('total_search_click').alias('user_fashion_search_click_sum'),
    pl.mean('total_search_click').alias('user_fashion_search_click_mean'),
    pl.n_unique('content_id_hashed').alias('user_fashion_content_count_searched')
).with_columns(
    (pl.col('user_fashion_search_click_sum') / pl.col('user_fashion_search_impression_sum')).fill_nan(0).alias('user_fashion_search_ctr_sum'),
    (pl.col('user_fashion_search_click_mean') / pl.col('user_fashion_search_impression_mean')).fill_nan(0).alias('user_fashion_search_ctr_mean')
).collect()

user_fashion_sitewide_agg = user_fashion_sitewide_log_lazy.group_by('user_id_hashed').agg(
    pl.sum('total_click').alias('user_fashion_sitewide_click_sum'),
    pl.mean('total_click').alias('user_fashion_sitewide_click_mean'),
    pl.sum('total_cart').alias('user_fashion_sitewide_cart_sum'),
    pl.mean('total_cart').alias('user_fashion_sitewide_cart_mean'),
    pl.sum('total_fav').alias('user_fashion_sitewide_fav_sum'),
    pl.mean('total_fav').alias('user_fashion_sitewide_fav_mean'),
    pl.sum('total_order').alias('user_fashion_sitewide_order_sum'),
    pl.mean('total_order').alias('user_fashion_sitewide_order_mean'),
    pl.n_unique('content_id_hashed').alias('user_fashion_content_count_site')
).with_columns(
    (pl.col('user_fashion_sitewide_cart_sum') / pl.col('user_fashion_sitewide_click_sum')).fill_nan(0).alias('user_fashion_cart_to_click_ratio_sum'),
    (pl.col('user_fashion_sitewide_cart_mean') / pl.col('user_fashion_sitewide_click_mean')).fill_nan(0).alias('user_fashion_cart_to_click_ratio_mean'),
    (pl.col('user_fashion_sitewide_order_sum') / pl.col('user_fashion_sitewide_cart_sum')).fill_nan(0).alias('user_fashion_order_to_cart_ratio_sum'),
    (pl.col('user_fashion_sitewide_order_mean') / pl.col('user_fashion_sitewide_cart_mean')).fill_nan(0).alias('user_fashion_order_to_cart_ratio_mean'),
    (pl.col('user_fashion_sitewide_fav_sum') / pl.col('user_fashion_sitewide_click_sum')).fill_nan(0).alias('user_fashion_fav_to_click_ratio_sum'),
    (pl.col('user_fashion_sitewide_fav_mean') / pl.col('user_fashion_sitewide_click_mean')).fill_nan(0).alias('user_fashion_fav_to_click_ratio_mean')
).collect()

# --- Statik Zaman Bazlı Özellikler ---
print("Statik zaman bazlı oranlar tüm veri üzerinden hesaplanıyor...")

user_sitewide_ratios = calculate_overall_ratios_pl(
    user_sitewide_log_lazy,
    date_col='ts_hour',
    group_by_cols=['user_id_hashed'],
    value_cols=['total_click', 'total_cart', 'total_order', 'total_fav']
).collect()

content_sitewide_ratios = calculate_overall_ratios_pl(
    content_sitewide_log_lazy,
    date_col='date',
    group_by_cols=['content_id_hashed'],
    value_cols=['total_click', 'total_cart', 'total_order', 'total_fav']
).collect()

# --- Kullanıcı Aktivite Süresi Özellikleri ---
print("Kullanıcı aktivite süresi özellikleri ekleniyor...")

user_activity_dates = pl.concat([
    user_search_log_lazy.select(['user_id_hashed', 'ts_hour']),
    user_sitewide_log_lazy.select(['user_id_hashed', 'ts_hour']),
    user_top_terms_log_lazy.select(['user_id_hashed', 'ts_hour']),
    user_fashion_search_log_lazy.select(['user_id_hashed', 'ts_hour']),
    user_fashion_sitewide_log_lazy.select(['user_id_hashed', 'ts_hour'])
]).group_by('user_id_hashed').agg([
    pl.min('ts_hour').alias('user_first_seen_date'),
    pl.max('ts_hour').alias('user_last_seen_date')
]).collect()

max_date = user_activity_dates.select(pl.col('user_last_seen_date').max()).item()

user_activity_features = user_activity_dates.with_columns([
    (pl.col('user_last_seen_date') - pl.col('user_first_seen_date')).dt.total_days().alias('user_activity_duration_days'),
    (max_date - pl.col('user_last_seen_date')).dt.total_days().alias('days_since_last_activity')
])

# --- Ürün Yaş Kategorisi Özelliği ---
print("Ürün yaş kategorisi özelliği ekleniyor...")

max_content_date = content_metadata.select(pl.col('content_creation_date').max()).item()

content_age_features = content_metadata.with_columns([
    (max_content_date - pl.col('content_creation_date')).dt.total_days().alias('content_age_days_raw')
]).with_columns([
    pl.when(pl.col('content_age_days_raw') <= 30).then(1)
    .when(pl.col('content_age_days_raw') <= 90).then(2)
    .when(pl.col('content_age_days_raw') <= 180).then(3)
    .when(pl.col('content_age_days_raw') <= 365).then(4)
    .when(pl.col('content_age_days_raw') <= 730).then(5)
    .when(pl.col('content_age_days_raw') <= 1095).then(6)
    .when(pl.col('content_age_days_raw') <= 1460).then(7)
    .when(pl.col('content_age_days_raw') <= 1825).then(8)
    .otherwise(9)
    .alias('content_age_category'),
    pl.col('content_age_days_raw').alias('content_age_days')
]).select(['content_id_hashed', 'content_age_days', 'content_age_category'])

# --- Güncellenmiş Birleştirme Planı ---
def build_merge_plan(df_lazy, is_hourly):
    """
    Tüm birleştirmeleri yapan ve yeni özellikleri ekleyen birleştirme planı.
    """
    if is_hourly:
        df_lazy = add_hourly_features(df_lazy, 'ts_hour')
    else:
        df_lazy = add_daily_features(df_lazy, 'session_date')
        
    content_metadata_processed = content_metadata.with_columns(
        pl.col("content_creation_date").dt.cast_time_unit("ms").dt.replace_time_zone(None)
    ).lazy()

    # price_data'yı birleştirmeye hazırlık: Günlük gruplama yap ve oranları ekle
    price_features_daily = content_price_data_lazy.with_columns([
        pl.col('update_date').cast(pl.Date).alias('update_date_daily')
    ]).group_by(['content_id_hashed', 'update_date_daily']).agg([
        pl.mean('original_price').alias('content_daily_original_price_mean'),
        pl.mean('selling_price').alias('content_daily_selling_price_mean'),
        pl.mean('discounted_price').alias('content_daily_discounted_price_mean'),
        pl.mean('content_rate_avg').alias('content_daily_content_rate_avg_mean'),
        pl.mean('content_review_count').alias('content_daily_review_count_mean'),
        pl.mean('content_rate_count').alias('content_daily_rate_count_mean')
    ]).lazy()

    # Yeni oranları hesapla
    price_features_with_ratios = add_price_ratios_pl(price_features_daily)
    
    return (
        df_lazy
        .with_columns(
            pl.col('ts_hour').cast(pl.Date).alias('session_date')
        )
        .join(content_metadata_processed, on='content_id_hashed', how='left')
        .join(content_search_agg.lazy(), on='content_id_hashed', how='left')
        .join(content_sitewide_agg.lazy(), on='content_id_hashed', how='left')
        .join(content_term_stats.lazy(), on='content_id_hashed', how='left')
        
        # Gün bazlı birleştirmeyi yap
        .join(
            price_features_with_ratios,
            left_on=['content_id_hashed', 'session_date'],
            right_on=['content_id_hashed', 'update_date_daily'],
            how='left'
        )
        
        .join(content_sitewide_ratios.lazy(), on='content_id_hashed', how='left')
        .join(content_age_features.lazy(), on='content_id_hashed', how='left')
        .join(user_metadata.lazy(), on='user_id_hashed', how='left')
        .join(user_search_agg.lazy(), on='user_id_hashed', how='left')
        .join(user_sitewide_agg.lazy(), on='user_id_hashed', how='left')
        .join(user_top_terms_agg.lazy(), on=['user_id_hashed', 'search_term_normalized'], how='left')
        .join(user_fashion_search_agg.lazy(), on='user_id_hashed', how='left')
        .join(user_fashion_sitewide_agg.lazy(), on='user_id_hashed', how='left')
        .join(user_sitewide_ratios.lazy(), on='user_id_hashed', how='left')
        .join(user_activity_features.lazy().drop(['user_first_seen_date', 'user_last_seen_date']), on='user_id_hashed', how='left')
        .join(term_popularity_agg.lazy(), on='search_term_normalized', how='left')
        .with_columns([
            (pl.col("ts_hour").dt.cast_time_unit("ms").dt.replace_time_zone(None) - pl.col("content_creation_date")).dt.total_days().cast(pl.Int32).alias("content_age_days_from_session")
        ])
    )

# --- Post-Merge Özellik Mühendisliği ---
def add_post_merge_features(df):
    """Birleştirme sonrası özellikler ekler"""
    return df.with_columns([
        # Zaman bazlı etkileşim özellikleri
        (pl.col("ts_hour") - pl.col("user_last_interaction_date")).dt.seconds().alias("seconds_since_user_last_interaction"),
        (pl.col("ts_hour") - pl.col("content_last_interaction_date")).dt.seconds().alias("seconds_since_content_last_interaction"),
        
        # Kullanıcı/ürün aktivite yoğunluğu ile mevcut zamanın ilişkisi
        (pl.col("hour_of_day") * pl.col("hourly_activity_density")).alias("hour_activity_score"),
        (pl.col("day_of_week") * pl.col("daily_activity_density")).alias("day_activity_score"),
        
        # Fiyat oranları
        ((pl.col("content_daily_original_price_mean") - pl.col("content_daily_selling_price_mean")) / pl.col("content_daily_original_price_mean")).fill_nan(0).alias("discount_rate"),
        (pl.col("content_daily_selling_price_mean") / pl.col("content_daily_original_price_mean")).fill_nan(0).alias("selling_to_original_ratio"),
        
        # Etkileşim oranları
        (pl.col("total_cart_total") / pl.col("total_click_total")).fill_nan(0).alias("cart_to_click_ratio"),
        (pl.col("total_order_total") / pl.col("total_cart_total")).fill_nan(0).alias("order_to_cart_ratio")
    ])

train_sessions_lazy = lazy_load_parquet("train_sessions.parquet")
test_sessions_lazy = lazy_load_parquet("test_sessions.parquet")

# Birleştirme planını uygula
train_sessions_final_lazy = build_merge_plan(train_sessions_lazy, is_hourly=True)
test_sessions_final_lazy = build_merge_plan(test_sessions_lazy, is_hourly=True)

# Lazy DataFrame'leri eager DataFrame'e dönüştürme ve işlemlerin tamamlanması
train_sessions = train_sessions_final_lazy.collect()
test_sessions = test_sessions_final_lazy.collect()

# --- Sonuçları Kontrol Etme ---
print("Tüm işlemler tamamlandı.")
print("Eğitim veri setinin boyutu (satır, sütun):", train_sessions.shape)
print("Test veri setinin boyutu (satır, sütun):", test_sessions.shape)


Veri setleri yükleniyor...
Zamansal özellikler ekleniyor...
Statik özellikler oluşturuluyor...
Statik zaman bazlı oranlar tüm veri üzerinden hesaplanıyor...
Kullanıcı aktivite süresi özellikleri ekleniyor...
Ürün yaş kategorisi özelliği ekleniyor...
Tüm işlemler tamamlandı.
Eğitim veri setinin boyutu (satır, sütun): (2773805, 153)
Test veri setinin boyutu (satır, sütun): (2988697, 149)


In [5]:
total_rows = train_sessions.height


for col in train_sessions.columns:
    null_count = train_sessions[col].null_count()
    pct = (null_count / total_rows) * 100
    print(f"{col}: {null_count} adet ({pct:.2f}%)")


ts_hour: 0 adet (0.00%)
search_term_normalized: 0 adet (0.00%)
clicked: 0 adet (0.00%)
ordered: 0 adet (0.00%)
added_to_cart: 0 adet (0.00%)
added_to_fav: 0 adet (0.00%)
user_id_hashed: 0 adet (0.00%)
content_id_hashed: 0 adet (0.00%)
session_id: 0 adet (0.00%)
hour_of_day: 0 adet (0.00%)
day_of_week: 0 adet (0.00%)
day_of_month: 0 adet (0.00%)
month_of_year: 0 adet (0.00%)
date_only: 0 adet (0.00%)
time_of_day: 0 adet (0.00%)
is_weekend: 0 adet (0.00%)
session_date: 0 adet (0.00%)
level1_category_name: 65 adet (0.00%)
level2_category_name: 65 adet (0.00%)
leaf_category_name: 65 adet (0.00%)
attribute_type_count: 18875 adet (0.68%)
total_attribute_option_count: 18875 adet (0.68%)
merchant_count: 18875 adet (0.68%)
filterable_label_count: 18875 adet (0.68%)
content_creation_date: 18875 adet (0.68%)
cv_tags: 580839 adet (20.94%)
content_total_search_impression_sum: 6269 adet (0.23%)
content_total_search_impression_mean: 6269 adet (0.23%)
content_total_search_click_sum: 6269 adet (0.23%)


In [6]:
import polars as pl

def clean_and_fill(df: pl.DataFrame, drop_nulls: bool = True) -> pl.DataFrame:
    # 1. Kritik sütunları null ise çıkar (opsiyonel)
    critical_cols = ["original_price", "selling_price", "discounted_price", "content_creation_date"]
    if drop_nulls:
        df = df.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in critical_cols if c in df.columns]))

    # 2. Kategorik sütunlar → fill_null("unknown") veya ""
    categorical_columns_with_unknown = ["user_gender"]
    categorical_columns_with_empty_string = [
        "level1_category_name",
        "level2_category_name",
        "leaf_category_name",
        "cv_tags"
    ]
    
    # user_gender için "unknown" ile doldurma
    for col in categorical_columns_with_unknown:
        if col in df.columns:
            df = df.with_columns(pl.col(col).fill_null("unknown"))

    # Diğer kategorik sütunlar için boş string ile doldurma
    for col in categorical_columns_with_empty_string:
        if col in df.columns:
            df = df.with_columns(pl.col(col).fill_null(""))

    # 3. Sayısal sütunlar → fill_null(0)
    numeric_cols = [c for c, dtype in zip(df.columns, df.dtypes) if dtype in (pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.Float32, pl.Float64)]
    # Kategorik olarak ele aldığımız sütunları sayısal doldurmadan çıkarıyoruz
    all_categorical_cols = categorical_columns_with_unknown + categorical_columns_with_empty_string
    for col in numeric_cols:
        if col not in all_categorical_cols:  # ekstra güvenlik
            df = df.with_columns(pl.col(col).fill_null(0))

    return df

# Kullanım
train_sessions = clean_and_fill(train_sessions, drop_nulls=False)
test_sessions = clean_and_fill(test_sessions, drop_nulls=False)

In [8]:
import polars as pl

# Kategorik sütunları bul (string veya kategorik tipte olanlar)
categorical_cols = [col for col in train_sessions.columns if train_sessions[col].dtype in [pl.Categorical, pl.Utf8]]

# Her sütun için unique sayısı ve ilk 5 değeri yazdır
for col in categorical_cols:
    unique_vals = train_sessions[col].unique()
    print(f"{col}: {unique_vals.len()} unique values")
    print("  First 5:", unique_vals.head(5).to_list())
    print("-" * 40)


search_term_normalized: 781 unique values
  First 5: ['abiye_elbise_kadin_tesettur', 'elbise_takim', 'ferace_tesettur', 'kimono_plaj', 'abiye_kadin']
----------------------------------------
user_id_hashed: 20996 unique values
  First 5: ['776264a65c743c6b', '2d8b47930c1ee000', 'd56b3b78730f6395', 'bea0784039762cd2', '20475ef80243a990']
----------------------------------------
content_id_hashed: 756156 unique values
  First 5: ['193c2b230800d1b0', '4fa8f94e6100bd7d', '1b53acf38b27c2a4', '8d1936999082a6ab', 'b391d956469ee3bf']
----------------------------------------
session_id: 21802 unique values
  First 5: ['train_b7d20a74383052f3', 'train_ac1d35ed5c25628c', 'train_feccc9df8cbd02a8', 'train_6276a1e33777736f', 'train_d524c153af01a941']
----------------------------------------
level1_category_name: 17 unique values
  First 5: ['Süpermarket', '', 'Hobi & Eğlence', 'Kitap', 'Aksesuar']
----------------------------------------
level2_category_name: 98 unique values
  First 5: ['Bilgisayar

In [9]:
!pip install Unidecode -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.1 MB/s eta 0:00:0000:01


In [10]:
import polars as pl
from unidecode import unidecode
import re
import string

def clean_text_polars(text: pl.Expr) -> pl.Expr:
    """Polars Expression olarak metin temizleme işlemleri"""
    
    # Unidecode uygulama
    decoded = text.map_elements(lambda x: unidecode(x) if x is not None else "", return_dtype=pl.Utf8)
    
    # Tüm metin temizleme işlemlerini tek bir zincirde yapın
    cleaned = (
        decoded.str.replace_all(r'[' + re.escape(string.punctuation) + r']', ' ')
        .str.to_lowercase()
        .str.replace_all(r'\s+', ' ')
        .str.strip_chars()  # strip() yerine strip_chars() kullanılıyor
    )
    
    return cleaned

def preprocess_sessions_polars(df: pl.DataFrame) -> pl.DataFrame:
    """Polars ile combined category, cv_tags ve search_term temizlik işlemleri"""
    
    # 1️⃣ Combined Category Oluştur
    df = df.with_columns(
        pl.concat_str(
            [
                pl.col("level1_category_name").fill_null(""),
                pl.col("level2_category_name").fill_null(""),
                pl.col("leaf_category_name").fill_null("")
            ],
            separator="_"
        ).alias("combined_category")
    )
    
    # 2️⃣ Boş değerleri doldur
    df = df.with_columns(
        pl.col("cv_tags").fill_null(""),
        pl.col("combined_category").fill_null("")
    )
    
    # 3️⃣ Metin temizleme işlemleri
    text_cols = ["combined_category", "cv_tags"]
    if "search_term_normalized" in df.columns:
        text_cols.append("search_term_normalized")
    
    # Tüm metin sütunlarına temizleme uygula
    for col in text_cols:
        df = df.with_columns(
            clean_text_polars(pl.col(col)).alias(f"{col}_clean")
        )
    
    return df

# ==============================================
# Train ve Test Dataframe'leri için uygula
# ==============================================
train_sessions = preprocess_sessions_polars(train_sessions)
test_sessions = preprocess_sessions_polars(test_sessions)

In [11]:
import polars as pl

# --- Yardımcı Fonksiyonlar ---
def word_count(text: pl.Expr) -> pl.Expr:
    """Metindeki kelime sayısı (Polars Expression)"""
    # None -> "" ile doldurulmuş olmalı
    return text.str.split(by=" ").list.len()

def char_count(text: pl.Expr) -> pl.Expr:
    """Metindeki karakter sayısı (Polars Expression)"""
    return text.str.len_chars()  # UTF-8 uyumlu, Türkçe karakterleri doğru sayar

def create_label_encodings(train_df: pl.DataFrame, test_df: pl.DataFrame, cat_cols: list) -> dict:
    """Train + Test birleştirerek mapping sözlükleri üret (Polars uyumlu)"""
    mappings = {}
    for col in cat_cols:
        combined_vals = pl.concat([
            train_df[col].fill_null("").cast(pl.Utf8),
            test_df[col].fill_null("").cast(pl.Utf8)
        ]).unique()
        mappings[col] = {val: idx for idx, val in enumerate(combined_vals.to_list())}
    return mappings

def process_with_polars(df: pl.DataFrame, mappings: dict) -> pl.DataFrame:
    """Metin sütunlarını işleyip uzunluk + global label encoding özellikleri ekler"""
    text_cols = [
        "search_term_normalized_clean", 
        "combined_category_clean",  # Burada _final_ kısmını kaldırdık
        "cv_tags_clean"            # Burada da _final_ kısmını kaldırdık
    ]
    
    # Null değerleri boş string yap
    df = df.with_columns([pl.col(col).fill_null("") for col in text_cols])
    
    # Kelime + karakter sayısı feature'ları
    df = df.with_columns([
        word_count(pl.col("search_term_normalized_clean")).alias("search_term_word_count"),
        word_count(pl.col("combined_category_clean")).alias("category_word_count"),
        word_count(pl.col("cv_tags_clean")).alias("tags_word_count"),
        char_count(pl.col("search_term_normalized_clean")).alias("search_term_char_count"),
        char_count(pl.col("combined_category_clean")).alias("category_char_count"),
        char_count(pl.col("cv_tags_clean")).alias("tags_char_count"),
    ])
    
    # Global label encoding (replace kullanarak)
    for col, mapping in mappings.items():
        df = df.with_columns(
            pl.col(col)
            .fill_null("")
            .replace(mapping)
            .cast(pl.Int32)   # 👉 Label encoding sütununu integer yap
            .alias(f"{col}_label")
        )
    
    return df


# --- Kategori sütunları ---
cat_cols = [
    "level1_category_name",
    "level2_category_name",
    "leaf_category_name",
    "combined_category"  # Burada _clean eklemiyoruz çünkü orijinal sütunları kullanıyoruz
]

# --- Label encoding mapping oluştur ---
mappings = create_label_encodings(train_sessions, test_sessions, cat_cols)

# --- Uygula ---
train_sessions = process_with_polars(train_sessions, mappings)
test_sessions = process_with_polars(test_sessions, mappings)

In [12]:
import polars as pl
import numpy as np

def auto_downcast(df: pl.DataFrame, exclude: list[str] = None) -> pl.DataFrame:
    if exclude is None:
        exclude = []

    new_cols = []

    for col, dtype in zip(df.columns, df.dtypes):

        # 1️⃣ Exclude listesinde ise hiç dokunma
        if col in exclude:
            continue  

        # 2️⃣ int64 sütunlar → küçült
        if dtype == pl.Int64:
            col_min, col_max = df.select(
                [pl.min(col).alias("min"), pl.max(col).alias("max")]
            ).row(0)

            if np.iinfo(np.int8).min <= col_min and col_max <= np.iinfo(np.int8).max:
                target_type = pl.Int8
            elif np.iinfo(np.int16).min <= col_min and col_max <= np.iinfo(np.int16).max:
                target_type = pl.Int16
            elif np.iinfo(np.int32).min <= col_min and col_max <= np.iinfo(np.int32).max:
                target_type = pl.Int32
            else:
                target_type = pl.Int64

            new_cols.append(pl.col(col).cast(target_type).alias(col))

        # 3️⃣ float64 sütunlar → float32
        elif dtype == pl.Float64:
            new_cols.append(pl.col(col).cast(pl.Float32).alias(col))

        # 4️⃣ diğer tiplere dokunma (hiç ekleme)
        else:
            continue  

    # sadece dönüştürülen sütunları overwrite ediyoruz
    return df.with_columns(new_cols)




# Kullanım:
exclude_cols = ["ordered", "clicked", "added_to_fav", "added_to_cart"]

train_sessions = auto_downcast(train_sessions, exclude=exclude_cols)
test_sessions  = auto_downcast(test_sessions, exclude=exclude_cols)


# Baseline Model ve Skor hesaplama 

**skor hesaplaması için train val ayrımı yapılıyor skor hesaplama bize verdikleri trendyol_metric_group_auc modülü kullanarak hesaplanıyor**

# Submit Etme

In [14]:
# SHAP sonrası önemsiz (0 değerli) kolon listesi
drop_features = [
    "hour_of_day",
    "month_of_year",
    "time_of_day",
    "is_weekend",
    "attribute_type_count",
    "content_search_ctr_mean",
    "content_cart_to_click_ratio_mean",
    "content_order_to_cart_ratio_mean",
    "content_fav_to_click_ratio_mean",
    "content_term_ctr_mean",
    "user_search_ctr_mean",
    "user_cart_to_click_ratio_mean",
    "user_order_to_cart_ratio_mean",
    "user_fav_to_click_ratio_mean",
    "user_term_ctr_mean",
    "user_fashion_search_ctr_mean",
    "user_fashion_cart_to_click_ratio_mean",
    "user_fashion_order_to_cart_ratio_mean",
    "user_fashion_fav_to_click_ratio_mean",
    "term_ctr_mean",
]
train_sessions = train_sessions.drop(drop_features)
test_sessions = test_sessions.drop(drop_features)

In [15]:
import polars as pl
import numpy as np

# Veri setlerinizi yükledikten sonra bu kod bloğunu kullanın

# train_sessions verisindeki sonsuz değerleri 0 yapalım
for col in train_sessions.columns:
    if train_sessions[col].dtype in [pl.Float64, pl.Float32]:
        train_sessions = train_sessions.with_columns(
            pl.col(col).replace(np.inf, 0).replace(-np.inf, 0)
        )

# test_sessions verisindeki sonsuz değerleri 0 yapalım
for col in test_sessions.columns:
    if test_sessions[col].dtype in [pl.Float64, pl.Float32]:
        test_sessions = test_sessions.with_columns(
            pl.col(col).replace(np.inf, 0).replace(-np.inf, 0)
        )

# Eğer veride NaN (eksik) değerler de varsa, onları da 0 ile doldurabilirsiniz
# Bu, özellikle CatBoost gibi modeller için faydalı olabilir
train_sessions = train_sessions.fill_null(0)
test_sessions = test_sessions.fill_null(0)

In [16]:
!pip install pytorch-tabnet -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.6 MB/s eta 0:00:00:00:0100:01


In [ ]:
import polars as pl
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
from trendyol_metric_group_auc import score

# -----------------------------
# 1. TARGET ve FEATURE TANIMLARI
# -----------------------------
exclude_cols = {"added_to_cart", "added_to_fav", "clicked", "ordered"}
target_cols = ["ordered", "clicked", "added_to_cart", "added_to_fav"]

numeric_dtypes = (
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64
)

schema = train_sessions.schema
feature_cols = [
    col for col, dtype in schema.items()
    if dtype in numeric_dtypes and col not in exclude_cols
]

# -----------------------------
# 2. STRATIFIED TRAIN-VALID SPLIT
# -----------------------------
stratify_col = (
    train_sessions
    .select(pl.concat_str(target_cols, separator="_"))
    .to_series()
    .to_numpy()
)

train_idx, val_idx = train_test_split(
    np.arange(len(train_sessions)),
    test_size=0.2,
    stratify=stratify_col,
    random_state=42
)

train_df = train_sessions[train_idx]
val_df = train_sessions[val_idx]

train_X = train_df.select(feature_cols).to_numpy()
train_y = train_df.select("ordered").to_numpy().ravel()

val_X = val_df.select(feature_cols).to_numpy()
val_y = val_df.select("ordered").to_numpy().ravel()

# -----------------------------
# 3. TABNET MODELİ
# -----------------------------
tabnet_model = TabNetClassifier(
    n_d=32, n_a=32,   # decision ve attention katmanları
    n_steps=5,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='entmax', # sparse attention
    verbose=1
)

# -----------------------------
# 4. MODEL EĞİTİMİ
# -----------------------------
tabnet_model.fit(
    X_train=train_X, y_train=train_y,
    eval_set=[(val_X, val_y)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=5,
    patience=5,
    batch_size=1024,
    virtual_batch_size=128
)

# -----------------------------
# 5. VALIDATION PREDICTIONS
# -----------------------------
predictions = tabnet_model.predict_proba(val_X)[:, 1]

val_df = val_df.with_columns(
    pl.Series("prediction", predictions)
)

# -----------------------------
# 6. SUBMISSION ve SOLUTION
# -----------------------------
submission_df = (
    val_df.lazy()
    .sort("prediction", descending=True)
    .group_by("session_id")
    .agg([pl.col("content_id_hashed").alias("prediction_list")])
    .with_columns([pl.col("prediction_list").list.join(" ").alias("prediction")])
    .select(["session_id", "prediction"])
    .collect()
)

solution_df = (
    val_df.lazy()
    .group_by("session_id")
    .agg([
        pl.col("content_id_hashed").filter(pl.col("ordered") == 1).alias("ordered_items_list"),
        pl.col("content_id_hashed").filter(pl.col("clicked") == 1).alias("clicked_items_list"),
        pl.col("content_id_hashed").alias("all_items_list")
    ])
    .with_columns([
        pl.col("ordered_items_list").list.join(" ").alias("ordered_items"),
        pl.col("clicked_items_list").list.join(" ").alias("clicked_items"),
        pl.col("all_items_list").list.join(" ").alias("all_items")
    ])
    .select(["session_id", "ordered_items", "clicked_items", "all_items"])
    .collect()
)

# -----------------------------
# 7. SCORE
# -----------------------------
try:
    final_score = score(
        solution_df.to_pandas(),
        submission_df.to_pandas(),
        "session_id"
    )
    print(f"Validation Score: {final_score:.4f}")
except Exception as e:
    print("Skor hesaplanamadı:", e)

# -----------------------------
# 8. TEMİZLİK
# -----------------------------
del train_X, train_y, val_X, val_y
gc.collect()


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.02237 | val_auc: 0.69281 |  0:05:56s
epoch 1  | loss: 0.02044 | val_auc: 0.73325 |  0:11:48s
epoch 2  | loss: 0.02022 | val_auc: 0.73958 |  0:17:35s
epoch 3  | loss: 0.01986 | val_auc: 0.75977 |  0:23:22s
epoch 4  | loss: 0.01974 | val_auc: 0.7629  |  0:29:08s
Stop training because you reached max_epochs = 5 with best_epoch = 4 and best_val_auc = 0.7629


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [ ]:
import polars as pl
import numpy as np
import gc
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

# -----------------------------
# 1. VERİ VE FEATURE TANIMLARI
# -----------------------------
exclude_cols = {"added_to_cart", "added_to_fav", "clicked", "ordered"}
target_cols = ["ordered", "clicked", "added_to_cart", "added_to_fav"]

numeric_dtypes = (
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64
)

# Eğitim verisindeki feature kolonlarını tanımlıyoruz
train_schema = train_sessions.schema
feature_cols = [
    col for col, dtype in train_schema.items()
    if dtype in numeric_dtypes and col not in exclude_cols
]

# -----------------------------
# 2. TÜM EĞİTİM VERİSİ İLE MODELİ EĞİTME
# -----------------------------
print("TabNet modeli eğitiliyor...")
train_X = train_sessions.select(feature_cols).to_numpy()
train_y = train_sessions.select("ordered").to_numpy().ravel()

tabnet_model = TabNetClassifier(
    n_d=32, n_a=32,
    n_steps=5,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='entmax',
    verbose=1
)

tabnet_model.fit(
    X_train=train_X, y_train=train_y,
    max_epochs=5,
    batch_size=1024,
    virtual_batch_size=128
)

print("Model eğitimi tamamlandı. Şimdi test verisi üzerinde tahmin yapılacak.")

# -----------------------------
# 3. TEST VERİSİ ÜZERİNDE TAHMİN YAPMA
# -----------------------------
test_X = test_sessions.select(feature_cols).to_numpy()

# **Eğitilmiş modeli kullanarak** tahmin yapıyoruz
predictions = tabnet_model.predict_proba(test_X)[:, 1]

# Tahminleri test DataFrame'ine ekleyin
test_sessions = test_sessions.with_columns(
    pl.Series("prediction", predictions)
)

# Tahmin skorlarına göre her bir session_id için öğeleri sıralayın
test_sessions = test_sessions.sort(["session_id", "prediction"], descending=[False, True])

# -----------------------------
# 4. SUBMISSION DOSYASI OLUŞTURMA
# -----------------------------
# Her bir session_id için tahmin edilen content_id_hashed'leri birleştirin
submission_df = test_sessions.group_by("session_id").agg(
    pl.col("content_id_hashed").alias("prediction")
).with_columns(
    pl.col("prediction").list.join(" ")
)

# Sadece gerekli sütunları seçin ve dosyaya yazın
submission_df = submission_df.select(["session_id", "prediction"])

submission_df.write_csv("submission.csv")
print("Submission dosyası başarıyla oluşturuldu: submission.csv")

# -----------------------------
# 5. BELLEK TEMİZLİĞİ
# -----------------------------
del train_X, train_y, test_X, tabnet_model
gc.collect()

In [ ]:
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt

# ... (önceki kodlar aynen kalıyor) ...

# -----------------------------
# 4. FEATURE IMPORTANCE ANALİZİ
# -----------------------------

# Özellik önem sıralamasını al
feature_importance = tabnet_model.feature_importances_
feature_names = feature_cols

# Özellik adları ve önem değerlerini birleştir
importance_df = pl.DataFrame({
    "feature": feature_names,
    "importance": feature_importance
})

# Önem sırasına göre sırala
importance_df = importance_df.sort("importance", descending=True)

# İlk 30 özelliği
print("En önemli 30 özellik:")
print(importance_df.head(90).to_pandas().to_string(index=False))

# Son 30 özelliği
print("\nEn az önemli 30 özellik:")
print(importance_df.tail(90).to_pandas().to_string(index=False))
